# 3章 ニューラルネットワーク

このノートブックは、パーセプトロンからニューラルネットワークへの橋渡しを、解説とコードの両方で学んでいくものです。各セクションでは「なぜそうするのか」を言葉で説明し、すぐ下のセルで実際に動かして確かめられるようにしています。

**この章のゴール**

- パーセプトロンとニューラルネットワークの共通点・相違点を理解する
- 活性化関数（ステップ関数・シグモイド関数・ReLU）の意味と実装を知る
- 多次元配列（行列）の積を NumPy で計算できるようになる
- ニューラルネットワークの計算が「行列の積」で表せることを体感する

まずは、これ以降で使うライブラリを読み込んでおきましょう。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt



## 3.1 パーセプトロンからニューラルネットワークへ

前章までのパーセプトロンには、良い面と困った面がありました。

- **良い面**：複雑な処理であっても、原理的には表現できる（理論上の表現力が高い）
- **困った面**：望ましい出力を得るための「重み」を、人間が手作業で決めなければならない

ニューラルネットワークは、この「重みを適切に決める」という作業を、**データから自動で学習**できる点が大きな特長です。この章では、まずニューラルネットワークがどんな構造で、どのように信号を伝えるのかという仕組みに焦点を当てます。

### 3.1.1 ニューラルネットワークの例

ニューラルネットワークは層に分けて表します。図にすると、左から順に

- **入力層（第0層）**：入力 \(x_1, x_2, \dots\) を受け取る層
- **中間層（隠れ層・第1層）**：途中の計算を担う層。図には現れにくいので「隠れ層」とも呼ばれます
- **出力層（第2層）**：最終的な結果 \(y\) を出す層

という構造になっています。なお「第0層・第1層・第2層」という数え方は、後で Python で実装するときに添字と一致して都合が良いためです。

下の図は、入力層2ニューロン → 中間層3ニューロン → 出力層2ニューロンの、3層のネットワークを模式的に描いたものです（重みの線が「全結合」になっている様子に注目してください）。

In [ ]:
# ニューラルネットワークの層構造を図示する（説明用のスケッチ）
fig, ax = plt.subplots(figsize=(7, 4.5))

layers = {"input": (0, [0.7, -0.7]),
          "hidden": (1.6, [1.1, 0.0, -1.1]),
          "output": (3.2, [0.55, -0.55])}

# ノードの描画
coords = {}
for name, (x, ys) in layers.items():
    for i, y in enumerate(ys):
        coords[(name, i)] = (x, y)
        ax.scatter(x, y, s=1400, facecolor="white", edgecolor="black", zorder=3)

# 全結合のエッジを描画
def connect(a, b):
    for i in range(len(layers[a][1])):
        for j in range(len(layers[b][1])):
            x1, y1 = coords[(a, i)]
            x2, y2 = coords[(b, j)]
            ax.plot([x1, x2], [y1, y2], color="gray", lw=0.8, zorder=1)

connect("input", "hidden")
connect("hidden", "output")

ax.text(0, -1.6, "input (0)", ha="center")
ax.text(1.6, -1.6, "hidden (1)", ha="center")
ax.text(3.2, -1.6, "output (2)", ha="center")
ax.set_xlim(-0.6, 3.8); ax.set_ylim(-1.9, 1.6)
ax.axis("off")
ax.set_title("3-layer neural network")
plt.show()

### 3.1.2 パーセプトロンの復習

ニューラルネットワークの信号伝達を理解するために、まずパーセプトロンを思い出します。

入力 \(x_1, x_2\) を受け取り、出力 \(y\) を出すパーセプトロンは、次のように書けます。

$$
y =
\begin{cases}
0 & (b + w_1 x_1 + w_2 x_2 \le 0) \\
1 & (b + w_1 x_1 + w_2 x_2 > 0)
\end{cases}
\tag{3.1}
$$

ここで各記号の役割は次のとおりです。

- \(w_1, w_2\)：**重み**。各信号の重要度をコントロールするパラメータ
- \(b\)：**バイアス**。ニューロンの発火のしやすさをコントロールするパラメータ

重みとバイアスをまとめて入力の総和を計算し、その値が 0 を超えれば 1、超えなければ 0 を出力する、というのがパーセプトロンの動作です。

#### 関数を使ってすっきり書き直す

式(3.1)はやや冗長なので、**1つの関数** \(h(x)\) を導入して書き換えます。

$$
y = h(b + w_1 x_1 + w_2 x_2) \tag{3.2}
$$

$$
h(x) =
\begin{cases}
0 & (x \le 0) \\
1 & (x > 0)
\end{cases}
\tag{3.3}
$$

式(3.2)では「重み付き入力とバイアスの総和」を計算し、その結果を \(h(x)\) に渡して変換しています。この \(h(x)\) が、これから主役になる **活性化関数（activation function）** です。

#### 活性化関数の登場 — 2段階に分けて考える

式(3.2)の計算は、2段階に分けて書くとさらに見通しが良くなります。

$$
a = b + w_1 x_1 + w_2 x_2 \tag{3.4}
$$

$$
y = h(a) \tag{3.5}
$$

- まず式(3.4)で、重み付き入力とバイアスの総和 \(a\) を計算する
- 次に式(3.5)で、その \(a\) を活性化関数 \(h()\) に通して出力 \(y\) を得る

つまり活性化関数は、**入力信号の総和を、出力信号へ変換する**役割を持ちます。本書では、信号が流れるひとつひとつの点を「ニューロン」または「ノード」と呼びます。

## 3.2 活性化関数

パーセプトロンでは活性化関数として **ステップ関数**（しきい値0で0/1が切り替わる関数）を使っていました。ニューラルネットワークへ進む鍵は、ここで **別の活性化関数に置き換える**ことです。

この節では代表的な活性化関数をいくつか紹介し、比較していきます。

### 3.2.1 シグモイド関数

ニューラルネットワークでよく用いられる活性化関数のひとつが、**シグモイド関数（sigmoid function）** です。

$$
h(x) = \frac{1}{1 + \exp(-x)} \tag{3.6}
$$

ここで \(\exp(-x)\) は \(e^{-x}\) を意味し、\(e\) はネイピア数（約 2.7182…）です。式だけ見ると複雑そうですが、要するに「何らかの入力を与えると、何らかの出力が返ってくる関数」にすぎません。

たとえば次のように、具体的な値を入れると具体的な値が返ってきます。

- \(h(1.0) = 0.731 \dots )
- \(h(2.0) = 0.880\dots\)

ニューラルネットワークでは、この関数で変換した信号を次のニューロンへ伝えていきます。

### 3.2.2 ステップ関数の実装

シグモイドへ進む前に、まず比較対象の **ステップ関数**を Python で実装しておきます。素直に書くと次のようになります。

この実装は単純で分かりやすいのですが、引数 `x` が**実数（スカラー）しか受け取れない**という弱点があります。`step_function_scalar(3.0)` のように単一の値なら使えますが、`np.array([...])` のような NumPy 配列をまとめて渡すことはできません。

そこで、NumPy 配列にも対応できるように修正します。ポイントは次の2つです。

1. NumPy 配列に対して不等号 `x > 0` を行うと、各要素を比較した **真偽値（bool）の配列**が得られる
2. その bool 配列を `astype(np.int_)` で **整数型に変換**すると、`True → 1`、`False → 0` になる

### 3.2.3 ステップ関数のグラフ

ステップ関数がどんな形をしているか、`matplotlib` でグラフにしてみましょう。

`np.arange(-5.0, 5.0, 0.1)` は、\(-5.0\) から \(5.0\) まで \(0.1\) 刻みの NumPy 配列（\([-5.0, -4.9, \dots, 4.9]\)）を生成します。これを `step_function` に渡し、入力 `x` と出力 `y` をプロットします。

グラフのとおり、ステップ関数は **0を境にして出力が0から1へ階段状に切り替わる**関数です。

### 3.2.4 シグモイド関数の実装とグラフ

シグモイド関数（式3.6）はすでに `sigmoid()` として定義しました。この実装が NumPy 配列にもそのまま対応している点が重要です。

`np.exp(-x)` は配列の各要素に対して \(e^{-x}\) を計算してくれますし、スカラー `1` と NumPy 配列の割り算は **ブロードキャスト**によって要素ごとに行われます。そのため、配列を渡しても正しく計算できます。

### 3.2.5 シグモイド関数とステップ関数の比較

2つの関数を重ねて描くと、違いと共通点がはっきり見えてきます。

**相違点**

- **滑らかさ**：シグモイドは連続的に変化する滑らかな曲線。ステップ関数は0を境に出力が急に切り替わる
- **出力の値**：ステップ関数は0か1の2値だけ。シグモイドは0.731…や0.880…のような連続的な実数を出力する

この「連続的に変化して滑らかである」という性質は、ニューラルネットワークの**学習**において重要な意味を持ちます。

**共通点**

- 入力が小さいときは0に近く、入力が大きくなると1に近づく、という大まかな形は共通している
- どんな入力に対しても、出力は必ず **0から1の間**に収まる
- どちらも **非線形関数**である（次のセクションで詳しく見ます）

### 3.2.6 非線形関数

ステップ関数もシグモイド関数も、ともに **非線形関数**に分類されます。

- **線形関数**：出力が入力の定数倍になるような関数。\(h(x) = cx\)（\(c\)は定数）のように、グラフが1本の直線になる
- **非線形関数**：1本の直線では表せない関数。「線形でない」関数

ニューラルネットワークでは、活性化関数に**必ず非線形関数を使う**必要があります。なぜ線形関数ではダメなのでしょうか？

理由は、**線形関数を使うと層を深くする意味がなくなってしまう**からです。

たとえば線形な活性化関数 \(h(x) = cx\) を使って3層重ねたとします。出力は

$$
y = h(h(h(x))) = c \cdot c \cdot c \cdot x = c^3 x
$$

となり、これは結局 \(y = a x\)（ただし \(a = c^3\)）という **1層のネットワークと同じ**ものになってしまいます。つまり、何層重ねても「隠れ層のないネットワーク」と等価になり、層を深くする恩恵が消えてしまうのです。

だからこそ、層を重ねることの利点を活かすために、活性化関数には非線形関数を使う必要があります。

### 3.2.7 ReLU 関数

これまでステップ関数とシグモイド関数を見てきました。歴史的にはシグモイドが古くから使われてきましたが、最近のニューラルネットワークでは **ReLU（Rectified Linear Unit）** という関数が主に用いられます。

ReLU は、入力が0を超えていればその入力をそのまま出力し、0以下なら0を出力する、というシンプルな関数です。

$$
h(x) =
\begin{cases}
x & (x > 0) \\
0 & (x \le 0)
\end{cases}
\tag{3.7}
$$

実装は `np.maximum` を使えば一行で書けます。`np.maximum(0, x)` は、0 と入力 `x` を要素ごとに比較し、大きいほうを返します。

## 3.3 多次元配列の計算

ニューラルネットワークの計算は、**多次元配列（行列）の積**として効率的に書けます。ここでは NumPy で多次元配列を扱う方法をマスターし、その後ニューラルネットワークの実装につなげます。

### 3.3.1 多次元配列

多次元配列とは、簡単に言えば「数字の集合」です。1列に並んだもの（1次元配列）、長方形に並んだもの（2次元配列）、…と、N次元に並べたものを一般に多次元配列と言います。

配列の次元数は `np.ndim()`、形状（shape）は `.shape` で取得できます。

まず **1次元配列**から見てみましょう。

In [ ]:
A = np.array([1, 2, 3, 4])
print(A)
print("次元数 ndim :", np.ndim(A))
print("形状  shape:", A.shape)     # 結果はタプル (4,)
print("A[0]       :", A[0])

1次元配列でも形状は `(4,)` のように **タプル**で返ります。これは、2次元・3次元配列のときと結果の型を統一するためです。

次に **2次元配列（行列）**を作ってみます。2次元配列では、横方向の並びを **行（row）**、縦方向の並びを **列（column）** と呼びます。

In [ ]:
B = np.array([[1, 2], [3, 4], [5, 6]])
print(B)
print("次元数 ndim :", np.ndim(B))
print("形状  shape:", B.shape)     # (3, 2) → 3行2列の行列


### 3.3.2 行列の積

行列の積は、**左側の行列の「行」**と**右側の行列の「列」**を、要素ごとに掛けて足し合わせることで計算します。その結果が、対応する位置の要素になります。

たとえば \(2 ✖︎ 2\) の行列同士の積を考えると、結果の \((1,1)\) 要素は「左の1行目」と「右の1列目」の対応する要素を掛けて足したものになります。

NumPy では `np.dot(A, B)` で行列の積を計算できます（演算子 `@` も同じ意味です）。

#### 形状が異なる行列の積 — 「対応する次元の要素数を一致させる」

行列の積では、**左側の行列の1番目の次元（列数）**と、**右側の行列の0番目の次元（行数）**が一致している必要があります。一致していないとエラーになります。

たとえば \(2 \times 3\) の行列 \(A\) と \(3 \times 2\) の行列 \(B\) の積は計算でき、結果は \(2 \times 2\) になります。

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6]])     # 2 x 3
B = np.array([[1, 2], [3, 4], [5, 6]])   # 3 x 2
print("A.shape =", A.shape, " B.shape =", B.shape)
print("A の列数(3) と B の行数(3) が一致 → 積を計算できる")
print("np.dot(A, B) の shape =", np.dot(A, B).shape)   # (2, 2)


図にすると、積を取る2つの行列で「内側の次元（Aの列数とBの行数）」を一致させる必要があり、結果の行列は「外側の次元」の形になる、という関係です。

$$
\underbrace{A}_{m \times \mathbf{n}} \; \cdot \; \underbrace{B}_{\mathbf{n} \times p} \;=\; \underbrace{C}_{m \times p}
$$

内側の \(n\) が一致していればよく、結果 \(C\) は \(m \times p\) になります。

In [ ]:
# 2x3 と 3x4 の積 → 2x4 になる例
A = np.array([[1, 2, 3], [4, 5, 6]])              # 2 x 3
B = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])                   # 3 x 4
print("A:", A.shape, " B:", B.shape, " → A@B:", (A @ B).shape)
print(A @ B)

### 3.3.3 ニューラルネットワークの行列の積

いよいよ、NumPy の行列積を使って **ニューラルネットワークの計算**を行ってみましょう。

ここでは、バイアスと活性化関数を省略し、**重みだけ**を持つ単純なネットワークを考えます。入力 `X` と重み `W` の形を意識するのがポイントです。

In [ ]:
X = np.array([1, 2])
print("X =", X, "  X.shape =", X.shape)   # (2,)

W = np.array([[1, 3, 5],[2, 4, 6]])
print("W =")
print(W)
print("W.shape =", W.shape)               # (2, 3)

ここで形状を確認します。

- 入力 `X` の形状：\((2,)\) — 入力ニューロンが2つ
- 重み `W` の形状：\((2, 3)\) — 入力2つ × 出力3つ

行列積を成立させるには、**`X` の要素数（2）と `W` の0番目の次元（2）を一致させる**必要があります。これが一致しているので `np.dot(X, W)` が計算でき、結果 `Y` は3つの要素を持つ配列（出力ニューロン3つ分）になります。

In [ ]:
Y = np.dot(X, W)
print("Y =", Y, "  Y.shape =", Y.shape)   # (3,)
# Y[0] = 1*1 + 2*2 = 5
# Y[1] = 1*3 + 2*4 = 11
# Y[2] = 1*5 + 2*6 = 17

このように、`np.dot` を一度呼ぶだけで、出力ニューロン3つ分の重み付き総和を**まとめて**計算できました。

`for` ループで1つずつ計算する代わりに行列積を使うことで、ニューラルネットワークの計算を簡潔かつ効率的に書けるのが大きな利点です。

#### まとめ：1層分の順伝播

これまでの要素（重み付き総和 + バイアス + 活性化関数）を組み合わせると、1層分の計算は次のように書けます。

In [ ]:
# 入力2 → 出力3 の1層を、バイアスと活性化関数つきで計算してみる
X = np.array([1.0, 0.5])               # 入力 (2,)
W = np.array([[0.1, 0.3, 0.5],
              [0.2, 0.4, 0.6]])        # 重み (2, 3)
B = np.array([0.1, 0.2, 0.3])          # バイアス (3,)

A = np.dot(X, W) + B                    # 式(3.4)相当: 重み付き総和 + バイアス
Z = sigmoid(A)                         # 式(3.5)相当: 活性化関数に通す

print("総和 A =", A)
print("出力 Z =", Z)

**この章のまとめ**

- パーセプトロンとニューラルネットワークの主な違いは **活性化関数**にある。パーセプトロンはステップ関数、ニューラルネットワークはシグモイドや ReLU などの**滑らかな／非線形な**関数を使う
- 活性化関数に **非線形関数**を使うことで、層を深くする意味が生まれる
- ニューラルネットワークの計算は **行列の積（`np.dot`）** で効率よく表せる
- 行列積では、**対応する次元の要素数を一致させる**ことが鉄則

次章では、これらを組み合わせて多層のネットワーク全体の順伝播を実装していくことになります。